# Gradient Boosting Experiments — Credit Card Fraud Detection
### XGBoost, LightGBM, sklearn GradientBoosting + HistGradientBoosting
Same data split and preprocessing as the RF notebook. Goal: break past RF's recall ceiling of 0.84.

In [ ]:
!pip install xgboost lightgbm imbalanced-learn --quiet

## 1. Load & Preprocess Data (identical to RF notebook)

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    average_precision_score, precision_score, recall_score, f1_score
)

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'mlg-ulb/creditcardfraud',
    'creditcard.csv'
)

new_df = df.copy()
new_df['Amount'] = RobustScaler().fit_transform(new_df['Amount'].to_numpy().reshape(-1, 1))
new_df['Time']   = StandardScaler().fit_transform(new_df[['Time']])
new_df = new_df.sample(frac=1, random_state=42)

train, temp = train_test_split(new_df, test_size=0.2, stratify=new_df['Class'], random_state=42)
test,  val  = train_test_split(temp,   test_size=0.5, stratify=temp['Class'],   random_state=42)

x_train = train.drop(columns=['Class']); y_train = train['Class']
x_test  = test.drop(columns=['Class']);  y_test  = test['Class']
x_val   = val.drop(columns=['Class']);   y_val   = val['Class']

from sklearn.ensemble import RandomForestClassifier
rf_tmp = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_tmp.fit(x_train, y_train)
sorted_idx = np.argsort(rf_tmp.feature_importances_)[::-1]
important_features = x_train.columns[sorted_idx[:20]]

x_train_imp = x_train[important_features]
x_val_imp   = x_val[important_features]
x_test_imp  = x_test[important_features]

fraud_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

print(f'Train: {x_train_imp.shape}  |  Val: {x_val_imp.shape}  |  Test: {x_test_imp.shape}')
print(f'Fraud rate — train: {y_train.mean():.4%}  |  val: {y_val.mean():.4%}')
print(f'Negative/Positive ratio: {fraud_ratio:.0f}:1')
print(f'Top features: {list(important_features[:10])}')

## 2. Shared evaluation helper

In [ ]:
def threshold_tune(y_true, probs, name='Model'):
    """Sweep thresholds on PR curve; return F1-optimal and F2-optimal results."""
    prec, rec, thrs = precision_recall_curve(y_true, probs)
    n = len(thrs)
    eps = 1e-9
    f1 = np.array([2*prec[j+1]*rec[j+1] / (prec[j+1]+rec[j+1]+eps) for j in range(n)])
    f2 = np.array([5*prec[j+1]*rec[j+1] / (4*prec[j+1]+rec[j+1]+eps) for j in range(n)])

    j1 = np.argmax(f1)
    j2 = np.argmax(f2)

    results = {}
    for label, j in [('F1-optimal', j1), ('F2-optimal', j2)]:
        thr = thrs[j]
        preds = (probs >= thr).astype(int)
        p = precision_score(y_true, preds, zero_division=0)
        r = recall_score(y_true, preds, zero_division=0)
        f = f1_score(y_true, preds, zero_division=0)
        results[label] = {'thr': thr, 'P': p, 'R': r, 'F1': f}
        print(f'\n--- {name} ({label} threshold = {thr:.4f}) ---')
        print(classification_report(y_true, preds))

    pr_auc = average_precision_score(y_true, probs)
    print(f'PR-AUC (probability-based): {pr_auc:.4f}')
    results['pr_auc'] = pr_auc
    return results

## 3. Experiment 1 — XGBoost

In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight handles imbalance natively — no SMOTE needed
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=fraud_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
    early_stopping_rounds=30,
)

t0 = time.time()
xgb_model.fit(
    x_train_imp, y_train,
    eval_set=[(x_val_imp, y_val)],
    verbose=50
)
print(f'\nXGBoost fit time: {time.time()-t0:.1f}s')

xgb_probs = xgb_model.predict_proba(x_val_imp)[:, 1]
xgb_results = threshold_tune(y_val, xgb_probs, 'XGBoost (scale_pos_weight)')

## 4. Experiment 2 — XGBoost + SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

xgb_smote_pipe = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=42)),
    ('xgb', XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=1,
        random_state=42,
        n_jobs=-1,
        eval_metric='aucpr',
    ))
])

t0 = time.time()
xgb_smote_pipe.fit(x_train_imp, y_train)
print(f'XGBoost+SMOTE fit time: {time.time()-t0:.1f}s')

xgb_smote_probs = xgb_smote_pipe.predict_proba(x_val_imp)[:, 1]
xgb_smote_results = threshold_tune(y_val, xgb_smote_probs, 'XGBoost + SMOTE')

## 5. Experiment 3 — LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=fraud_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

t0 = time.time()
lgbm_model.fit(
    x_train_imp, y_train,
    eval_set=[(x_val_imp, y_val)],
)
print(f'\nLightGBM fit time: {time.time()-t0:.1f}s')

lgbm_probs = lgbm_model.predict_proba(x_val_imp)[:, 1]
lgbm_results = threshold_tune(y_val, lgbm_probs, 'LightGBM (scale_pos_weight)')

## 6. Experiment 4 — LightGBM + SMOTE

In [ ]:
lgbm_smote_pipe = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=42)),
    ('lgbm', LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=1,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ))
])

t0 = time.time()
lgbm_smote_pipe.fit(x_train_imp, y_train)
print(f'LightGBM+SMOTE fit time: {time.time()-t0:.1f}s')

lgbm_smote_probs = lgbm_smote_pipe.predict_proba(x_val_imp)[:, 1]
lgbm_smote_results = threshold_tune(y_val, lgbm_smote_probs, 'LightGBM + SMOTE')

## 7. Experiment 5 — sklearn HistGradientBoosting (no extra install needed)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
)

t0 = time.time()
hgb_model.fit(x_train_imp, y_train)
print(f'HistGradientBoosting fit time: {time.time()-t0:.1f}s')

hgb_probs = hgb_model.predict_proba(x_val_imp)[:, 1]
hgb_results = threshold_tune(y_val, hgb_probs, 'HistGradientBoosting (balanced)')

## 8. Comparison table — all gradient boosting experiments vs RF baseline

In [ ]:
rf_baseline = {'F1-optimal': {'P': 0.8913, 'R': 0.8367, 'F1': 0.8632, 'thr': 0.6752},
               'F2-optimal': {'P': 0.8913, 'R': 0.8367, 'F1': 0.8632, 'thr': 0.6752},
               'pr_auc': 0.746}

all_results = {
    'RF baseline (SMOTE 0.2)': rf_baseline,
    'XGBoost (scale_pos_weight)': xgb_results,
    'XGBoost + SMOTE': xgb_smote_results,
    'LightGBM (scale_pos_weight)': lgbm_results,
    'LightGBM + SMOTE': lgbm_smote_results,
    'HistGradientBoosting': hgb_results,
}

rows = []
for name, res in all_results.items():
    f1_opt = res['F1-optimal']
    f2_opt = res['F2-optimal']
    rows.append({
        'Model': name,
        'PR-AUC': res['pr_auc'],
        'P (F1-thr)': f1_opt['P'],
        'R (F1-thr)': f1_opt['R'],
        'F1 (F1-thr)': f1_opt['F1'],
        'P (F2-thr)': f2_opt['P'],
        'R (F2-thr)': f2_opt['R'],
        'F1 (F2-thr)': f2_opt['F1'],
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print()

best_f1_row = comparison.loc[comparison['F1 (F1-thr)'].idxmax()]
best_rec_row = comparison.loc[comparison['R (F2-thr)'].idxmax()]
print(f"Best F1:     {best_f1_row['Model']} — F1={best_f1_row['F1 (F1-thr)']:.4f}, R={best_f1_row['R (F1-thr)']:.4f}")
print(f"Best Recall: {best_rec_row['Model']} — R={best_rec_row['R (F2-thr)']:.4f}, F1={best_rec_row['F1 (F2-thr)']:.4f}")

## 9. Hyperparameter tuning for the best model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_param_dist = {
    'n_estimators': randint(200, 600),
    'max_depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'scale_pos_weight': [fraud_ratio * f for f in [0.5, 0.75, 1.0, 1.25, 1.5]],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'reg_alpha': [0, 0.5, 1, 2],
    'reg_lambda': [0.5, 1, 2, 5],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, n_jobs=-1, eval_metric='aucpr'),
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

t0 = time.time()
xgb_search.fit(x_train_imp, y_train)
print(f'\nXGBoost RandomizedSearch done in {time.time()-t0:.1f}s')
print(f'Best CV F1: {xgb_search.best_score_:.4f}')
print(f'Best params: {xgb_search.best_params_}')

xgb_tuned = xgb_search.best_estimator_
xgb_tuned_probs = xgb_tuned.predict_proba(x_val_imp)[:, 1]
xgb_tuned_results = threshold_tune(y_val, xgb_tuned_probs, 'XGBoost (tuned)')

## 10. Final evaluation on test set

In [ ]:
print('=== Final Test Set Evaluation ===')
print('Using best model from above with F1-optimal threshold from validation.\n')

# Pick whichever model performed best — adjust if a different model won
# Using the tuned XGBoost by default
final_model = xgb_tuned
final_probs_test = final_model.predict_proba(x_test_imp)[:, 1]

# Use the threshold found on validation
val_thr = xgb_tuned_results['F1-optimal']['thr']
final_preds = (final_probs_test >= val_thr).astype(int)

print(f'Threshold (from validation): {val_thr:.4f}')
print(classification_report(y_test, final_preds))
print(f'PR-AUC: {average_precision_score(y_test, final_probs_test):.4f}')